In [2]:

import os
import sys
from google import genai
from datetime import datetime
from typing import List, Dict, Any
from PIL import Image
import requests
import json
from google.genai import types # This is used to access the GenerateContentConfig class for setting system instructions and other configurations.
from pydantic import BaseModel


In [10]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
SCRIPT_FILE    = "../scripts/Shrek.pdf"       # <-- path to your script (.txt, .pdf, .docx)
OUTPUT_FILE    = "../parsed_scripts/parsed_script.json"  # <-- desired output path
MODEL          = "gpt-4o-mini"         # options: gpt-4o-mini, gpt-4o, gpt-3.5-turbo

### Test Request

In [4]:
gemini_client = genai.Client(api_key=GOOGLE_API_KEY, http_options={'timeout': 100_000})
response = gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=["You are a helpful assistant.", "Can you print 10 random words in a json format?"], # Contents = system + user messages
    config=types.GenerateContentConfig( # Pydantics parameters setter for controlling the response format and behavior of the model.
                                        response_mime_type="application/json",
)
)
print(response.text)

{
  "random_words": [
    "serendipity",
    "ephemeral",
    "mellifluous",
    "ubiquitous",
    "cacophony",
    "perspicacious",
    "halcyon",
    "quintessential",
    "effervescent",
    "luminous"
  ]
}


### Check Models available through gemini client

In [4]:
for model in gemini_client.models.list():
    if 'generateContent' in model.supported_actions:
        print(f"Model Name: {model.name}")

Model Name: models/gemini-2.5-flash
Model Name: models/gemini-2.5-pro
Model Name: models/gemini-2.0-flash
Model Name: models/gemini-2.0-flash-001
Model Name: models/gemini-2.0-flash-lite-001
Model Name: models/gemini-2.0-flash-lite
Model Name: models/gemini-2.5-flash-preview-tts
Model Name: models/gemini-2.5-pro-preview-tts
Model Name: models/gemma-3-1b-it
Model Name: models/gemma-3-4b-it
Model Name: models/gemma-3-12b-it
Model Name: models/gemma-3-27b-it
Model Name: models/gemma-3n-e4b-it
Model Name: models/gemma-3n-e2b-it
Model Name: models/gemini-flash-latest
Model Name: models/gemini-flash-lite-latest
Model Name: models/gemini-pro-latest
Model Name: models/gemini-2.5-flash-lite
Model Name: models/gemini-2.5-flash-image
Model Name: models/gemini-2.5-flash-lite-preview-09-2025
Model Name: models/gemini-3-pro-preview
Model Name: models/gemini-3-flash-preview
Model Name: models/gemini-3.1-pro-preview
Model Name: models/gemini-3.1-pro-preview-customtools
Model Name: models/gemini-3.1-fl

### Pydantic model for the expected output format of the parsed script lines

In [7]:
class ScriptLine(BaseModel):
    character: str
    line: str
    action: str
    page_number: int

### Upload the script file to Gemini's file storage

In [6]:
# Disclaimer: The following prompt; without the 100 line limit, was used with Gemini 2.5 Flashlite, to prevent API cost and maintain testing of free, 
# it can be tested here and completed in AI Studio for free: https://aistudio.google.com/models/gemini-2.5-flash-lite 
# example of the output given when running this code is given below.

file = gemini_client.files.upload(file=SCRIPT_FILE)

response = gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite", # Low cost model, good for parsing and extracting information from documents. 
    contents=[
                file,
                " Extract the first 5 character lines and the character that says them in this script as well as the page number the line is on. "
                "Format the output as a JSON array of objects with the following keys: character, line,action i.e.( speech, sound or act) page_number."
             ],
    config=types.GenerateContentConfig( # Pydantics parameters setter for controlling the response format and behavior of the model.
                                        temperature=0.0,
                                        response_mime_type="application/json", # Model's response is in JSON format
                                        system_instruction="You are a professional document analyst.",
                                        response_schema={"type": "array", #
                                                         "items": ScriptLine.model_json_schema() # Pydantics mold for the output format
                                                         },
                                      )
)

print(response.text)



ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\nPlease retry in 43.7331306s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{

In [ ]:
Output:

[
  {
    "character": "SHREK",
    "line": "(Narrator reading the book)\nOnce upon a time there was a lovely princess.",
    "action": "speech",
    "page_number": 1
  },
  {
    "character": "SHREK",
    "line": "(cont'd)\nBut she had an enchantment upon her of a fearful sort which could only be broken by love's first kiss.",
    "action": "speech",
    "page_number": 1
  },
  {
    "character": "SHREK",
    "line": "(cont'd)\nShe was locked away in a castle guarded by a terrible, fire breathing dragon.",
    "action": "speech",
    "page_number": 1
  },
  {
    "character": "SHREK",
    "line": "(cont'd)\nMany brave knights had attempted to free her from this dreadful prison, but none prevailed.",
    "action": "speech",
    "page_number": 1
  },
  {
    "character": "SHREK",
    "line": "(cont'd)\nShe waited in the Dragon's Keep, in the highest room of the tallest tower, for her true love and true love's first kiss--(beat)occare--c)",
    "action": "speech",
    "page_number": 1
  },
  {
    "character": "SHREK",
    "line": "(cont'd)\nYeah -- Like that's ever gonna happen!",
    "action": "speech",
    "page_number": 1
  },
  {
    "character": "SHREK",
    "line": "(good natured)\nWhat a load of sh--",
    "action": "speech",
    "page_number": 1
  },
  {
    "character": "SHREK",
    "line": "Shrek comes out of the out-house. He picks his underwear from his butt and notices a page of the book stuck to his foot. He shakes it off. Looking up he admires his house.",
    "action": "act",
    "page_number": 2
  },
  {
    "character": "SHREK",
    "line": "Bucket going into mud. Shrek is having a mud shower.",
    "action": "act",
    "page_number": 2
  },
  {
    "character": "SHREK",
    "line": "Shrek grabs a bug from his toiletry jar, squeezes it on to a bone a proceeds to clean his teeth. He checks his smile and the mirror cracks, making him even happier.",
    "action": "act",
    "page_number": 2
  },
  {
    "character": "SHREK",
    "line": "Shrek jumps into a mucky swamp pond, we see the water bubbling and a relieved look on Shrek's face. Dead fish float belly up, Shrek adds them to his shopping bag.",
    "action": "act",
    "page_number": 2
  },
  {
    "character": "SHREK",
    "line": "Shrek leaves the pond, his legs are covered in leeches, he pulls one off and tastes it. It's good.",
    "action": "act",
    "page_number": 2
  },
  {
    "character": "SHREK",
    "line": "Mud secretes out of a hollowed log floating inside the pond. Shrek pops out behind it and grabs his \"Mud Squid\" from the muddy pile.",
    "action": "act",
    "page_number": 2
  },
  {
    "character": "SHREK",
    "line": "Shrek is painting. Delicately creating what seems to be a beautiful vista. He takes the painting from the easel and hammers a stick to the back. As he walks off with it we see that it's a \"Beware\" sign.",
    "action": "act",
    "page_number": 2
  },
  {
    "character": "SHREK",
    "line": "Shrek enters his home.",
    "action": "act",
    "page_number": 2
  },
  {
    "character": "SHREK",
    "line": "Pub door with \"Wanted Creatures Reward\" sign bursts open and angry mob pours out. An ogre hunter draws a plan of attack onto ground. Villagers reach for torches and pitchforks.",
    "action": "act",
    "page_number": 3
  },
  {
    "character": "SHREK",
    "line": "Shrek grabs spoon and begins to eat his dinner.",
    "action": "act",
    "page_number": 3
  },
  {
    "character": "SHREK",
    "line": "Silhouette of mob against the setting sky running into the forest.",
    "action": "act",
    "page_number": 3
  },
  {
    "character": "SHREK",
    "line": "Shrek strikes a match and starts fire in hearth with a burp.",
    "action": "act",
    "page_number": 3
  },
  {
    "character": "SHREK",
    "line": "Angry mob raise their torches and pitchforks.",
    "action": "act",
    "page_number": 3
  }
  .
  .
  .
  .
  

In [12]:
# Parse the response JSON and save to output file
parsed_lines = json.loads(response.text)

with open(OUTPUT_FILE, "w") as f:
    json.dump(parsed_lines, f, indent=2)

print(f"Saved {len(parsed_lines)} script lines to {OUTPUT_FILE}")

Saved 1 script lines to ../parsed_scripts/parsed_script.json


### Cleanup (deletes the file from the cloud immediately)

In [ ]:
gemini_client.files.delete(name=file.name)  